<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [5]</a>'.</span>

# Table of Contents

1. [Introduction](#tampered-image-detection-and-localization---submission-notebook-vk7)
2. [Project Objectives](#project-objectives-fulfilled-vs-remaining)
3. [Environment Setup](#1-environment-setup)
    - 1.1 Runtime Configuration and Kaggle-Style Directories
    - 1.2 Dataset Discovery Helpers
    - 1.3 Dataset Resolution from Google Drive or Kaggle
    - 1.4 Suppress libpng Warnings
4. [Dataset Explanation](#2-dataset-explanation)
    - 2.1 Discover Dataset Root and Build Metadata
    - 2.2 Train / Validation / Test Split
5. [Data Loading and Preprocessing](#3-data-loading-and-preprocessing)
6. [Prior Experiment Block (Source-Preserved)](#source-preserved-prior-experiment-block)
    - 3.1 Package Installation and Warning Suppression
    - 3.2 Imports
    - 3.3 Metadata Loading
    - 3.4 Image and Mask Transforms
    - 3.5 Dataset Class Definition
    - 3.6 U-Net with Classification Head
    - 3.7 DataLoader Construction
    - 3.8 Training Loop and Validation
7. [Model Architecture](#4-model-architecture)
8. [Training Strategy](#5-training-strategy)
9. [Hyperparameters](#6-hyperparameters)
10. [Experiment Tracking (W&B)](#7-experiment-tracking)
11. [Effective Submission Training Loop](#8-effective-submission-training-loop)
    - 8.1 Dependencies and Imports
    - 8.2 Metadata Loading
    - 8.3 Image and Mask Transforms
    - 8.4 Dataset Class
    - 8.5 U-Net with Classifier Architecture
    - 8.6 DataLoader Construction
    - 8.7 Loss Functions and Reporting Metrics
    - 8.8 Training History and Metric Helpers
    - 8.9 Training and Evaluation Routines
    - 8.10 Training Loop Execution
    - 8.11 Final Test Evaluation
12. [Training Curves](#81-training-curves-and-learning-behavior)
13. [Evaluation Methodology](#9-evaluation-methodology)
14. [Visualization of Predictions](#10-visualization-of-predictions)
    - 10.1 Sample Collection for Qualitative Review
    - 10.2 Submission-Ready Prediction Panels
15. [Conclusion](#conclusion)

# Tampered Image Detection and Localization - Submission Notebook (vK.7)

This Google Colab notebook presents a complete assignment submission for tampered image detection and tampered region localization.
The workflow keeps the original implementation intact while improving readability, traceability, and presentation quality for final review.

**Notebook deliverables demonstrated here**
- image-level tamper detection through the classifier head
- pixel-level tampered region localization through the segmentation branch
- reproducible Colab execution using the preserved source notebook workflow
- qualitative visual evidence showing predicted masks and overlays

**Assignment alignment:** This notebook addresses the full assignment pipeline from dataset preparation to tamper detection, localization, evaluation, and qualitative reporting.

## Project Objectives: Fulfilled vs Remaining

The table below maps the notebook output to the original assignment expectations so the reader can quickly verify coverage.

| Status | Objective | Notes |
|---|---|---|
| Fulfilled | Detect whether an image is tampered | Implemented by the classifier head and reported through image-level accuracy. |
| Fulfilled | Localize tampered regions | Implemented by the segmentation output and shown with predicted masks and overlays. |
| Fulfilled | Run in one Colab notebook | Environment setup, dataset preparation, training, evaluation, and visualization remain in one notebook. |
| Fulfilled | Track experiments | W&B tracking is added for the effective submission run. |
| Remaining | Broader robustness benchmarking | Additional corruption, cross-dataset, and stress testing are still future work. |
| Remaining | Larger-scale validation | The notebook remains a focused assignment artifact, not a production benchmark suite. |
| Remaining | Deployment hardening | Packaging, serving, and latency optimization are not part of this submission notebook. |
| Remaining | Richer failure analysis | More systematic error analysis can be added in future iterations. |

## 1. Environment Setup

The next cells configure the Colab runtime, emulate the Kaggle-style directory structure used by the source notebook, and prepare the dataset so the original code can run without redesigning the implementation.

**Section breakdown**
- runtime package installation and Kaggle-style working directories
- dataset discovery from Google Drive or Kaggle download

**Assignment alignment:** This section supports the assignment requirement that the end-to-end tampering workflow runs in a single notebook environment.

### 1.1 Runtime Configuration and Kaggle-Style Directories

Installs required packages for the Colab runtime and creates the Kaggle-style directory structure
(`/kaggle/input`, `/kaggle/working`) so that source notebook path references work without modification.

In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
KAGGLE_DATASET_SLUG = "sagnikkayalcse52/casia-spicing-detection-localization"
COLAB_KAGGLE_INPUT = Path("/kaggle/input")
COLAB_KAGGLE_WORKING = Path("/kaggle/working")

if IN_COLAB:
    subprocess.check_call(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "albumentations==1.3.1",
            "opencv-python-headless==4.10.0.84",
            "kaggle",
        ]
    )

COLAB_KAGGLE_INPUT.mkdir(parents=True, exist_ok=True)
COLAB_KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)

print("IN_COLAB:", IN_COLAB)
print("COLAB_KAGGLE_INPUT:", COLAB_KAGGLE_INPUT)
print("COLAB_KAGGLE_WORKING:", COLAB_KAGGLE_WORKING)
print("KAGGLE_DATASET_SLUG:", KAGGLE_DATASET_SLUG)


IN_COLAB: False
COLAB_KAGGLE_INPUT: \kaggle\input
COLAB_KAGGLE_WORKING: \kaggle\working
KAGGLE_DATASET_SLUG: sagnikkayalcse52/casia-spicing-detection-localization


### 1.2 Dataset Discovery Helpers

Utility functions that validate dataset layout, search Google Drive for existing copies,
normalize paths into the Kaggle-style directory, and fall back to a Kaggle API download when necessary.

#### 1.2.1 Layout Validation and Path Utilities

These helpers check whether a directory contains the expected `IMAGE` and `MASK` subdirectories
and deduplicate discovered paths for deterministic selection.

In [2]:
import os
import shutil
from pathlib import Path


TARGET_DATASET_DIR = COLAB_KAGGLE_INPUT / "casia-spicing-detection-localization"
DRIVE_SEARCH_ROOTS = [
    Path("/content/drive/MyDrive"),
    Path("/content/drive/Shareddrives"),
]


def has_image_and_mask_dirs(path: Path) -> bool:
    """
    Purpose:
        Check whether a directory already matches the expected dataset layout.

    Inputs:
        path (Path): Candidate directory to validate.

    Returns:
        bool: True when the directory contains both IMAGE and MASK subdirectories.

    Notes:
        The notebook expects the CASIA-style folder structure before metadata generation can begin.
    """
    if path is None or not path.exists() or not path.is_dir():
        return False
    try:
        child_names = {child.name.lower() for child in path.iterdir() if child.is_dir()}
    except OSError:
        return False
    return "image" in child_names and "mask" in child_names


def sorted_unique_paths(paths):
    """
    Purpose:
        Deduplicate and sort discovered filesystem paths before selecting a dataset root.

    Inputs:
        paths (Iterable[Path]): Candidate paths collected during recursive search.

    Returns:
        list[Path]: Unique paths ordered to prefer shorter, cleaner directory matches.

    Notes:
        Sorting makes the automatic selection step deterministic across repeated notebook runs.
    """
    unique = []
    seen = set()
    for path in paths:
        key = str(path.resolve()) if path.exists() else str(path)
        if key in seen:
            continue
        seen.add(key)
        unique.append(path)
    return sorted(unique, key=lambda p: (len(p.parts), str(p).lower()))



#### 1.2.2 Google Drive Dataset Search

Recursively searches mounted Google Drive directories for a dataset folder
that matches the CASIA naming convention and contains the expected layout.

In [3]:
def find_dataset_in_drive(search_roots):
    """
    Purpose:
        Search mounted Google Drive locations for a dataset directory containing IMAGE and MASK folders.

    Inputs:
        search_roots (Iterable[Path]): Top-level Drive directories to search recursively.

    Returns:
        Path | None: The best matching dataset directory, or None if nothing suitable is found.

    Notes:
        The search prefers directories whose names already resemble the expected dataset slug.
    """
    preferred = []
    fallback = []
    for root in search_roots:
        if not root.exists():
            continue

        if has_image_and_mask_dirs(root):
            preferred.append(root)

        for pattern in ["casia-spicing-detection-localization", "*casia*"]:
            for candidate in root.rglob(pattern):
                if candidate.is_dir() and has_image_and_mask_dirs(candidate):
                    preferred.append(candidate)

        if preferred:
            continue

        for candidate in root.rglob("*"):
            if candidate.is_dir() and has_image_and_mask_dirs(candidate):
                fallback.append(candidate)

    candidates = sorted_unique_paths(preferred or fallback)
    return candidates[0] if candidates else None



#### 1.2.3 Dataset Normalization and Kaggle API Fallback

`normalize_dataset_dir` symlinks or copies the discovered dataset into the Kaggle-style input path.
`ensure_dataset_from_kaggle` downloads the dataset through the Kaggle API when Drive is unavailable.

In [4]:
def normalize_dataset_dir(source_dir: Path, target_dir: Path) -> Path:
    """
    Purpose:
        Mirror the discovered dataset into the Kaggle-style input path expected by later notebook cells.

    Inputs:
        source_dir (Path): The dataset directory found in Drive or a download cache.
        target_dir (Path): The normalized Kaggle-style destination directory.

    Returns:
        Path: The directory that should be used as the canonical dataset root.

    Notes:
        The function first tries to create a symlink and falls back to copying when symlinks are unavailable.
    """
    target_dir.parent.mkdir(parents=True, exist_ok=True)

    if target_dir.exists() and has_image_and_mask_dirs(target_dir):
        return target_dir

    if target_dir.is_symlink():
        target_dir.unlink()
    elif target_dir.exists():
        shutil.rmtree(target_dir)

    try:
        os.symlink(source_dir, target_dir, target_is_directory=True)
        print(f"Symlinked dataset: {target_dir} -> {source_dir}")
    except OSError:
        shutil.copytree(source_dir, target_dir, dirs_exist_ok=True)
        print(f"Copied dataset into: {target_dir}")
    return target_dir


def ensure_dataset_from_kaggle(target_dir: Path) -> Path | None:
    """
    Purpose:
        Download the dataset through the Kaggle API when Google Drive does not provide a usable copy.

    Inputs:
        target_dir (Path): Destination path where the normalized dataset should be exposed.

    Returns:
        Path | None: The normalized dataset directory if the download succeeds, otherwise None.

    Notes:
        This fallback keeps the rest of the notebook unchanged by recreating the expected folder layout.
    """
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
    except Exception as exc:
        print(f"Kaggle API import failed: {exc}")
        return None

    download_root = COLAB_KAGGLE_INPUT / "_downloads"
    download_root.mkdir(parents=True, exist_ok=True)

    try:
        api = KaggleApi()
        api.authenticate()
        api.dataset_download_files(KAGGLE_DATASET_SLUG, path=str(download_root), unzip=True, quiet=False)
    except Exception as exc:
        print(f"Kaggle download failed: {exc}")
        return None

    candidates = []
    if has_image_and_mask_dirs(download_root):
        candidates.append(download_root)
    for candidate in download_root.rglob("*"):
        if candidate.is_dir() and has_image_and_mask_dirs(candidate):
            candidates.append(candidate)

    candidates = sorted_unique_paths(candidates)
    if not candidates:
        return None

    return normalize_dataset_dir(candidates[0], target_dir)



### 1.3 Dataset Resolution from Google Drive or Kaggle

Attempts to mount Google Drive and locate an existing dataset copy.
Falls back to the Kaggle API download when Drive-based discovery fails.
Raises an error if no valid dataset can be prepared.

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [5]:
dataset_dir = None

try:
    from google.colab import drive

    # Prefer a mounted Google Drive copy so the notebook can reuse existing data without re-downloading.
    drive.mount("/content/drive", force_remount=False)
    dataset_dir = find_dataset_in_drive(DRIVE_SEARCH_ROOTS)
    if dataset_dir is not None:
        dataset_dir = normalize_dataset_dir(dataset_dir, TARGET_DATASET_DIR)
        print(f"Using dataset from Google Drive: {dataset_dir}")
except Exception as exc:
    print(f"Google Drive mount/search skipped: {exc}")

if dataset_dir is None:
    # Fall back to the Kaggle API only when Drive-based discovery does not find a valid dataset root.
    dataset_dir = ensure_dataset_from_kaggle(TARGET_DATASET_DIR)
    if dataset_dir is not None:
        print(f"Using dataset from Kaggle download: {dataset_dir}")

if dataset_dir is None or not has_image_and_mask_dirs(dataset_dir):
    raise FileNotFoundError(
        "Could not prepare /kaggle/input/casia-spicing-detection-localization. "
        "Provide a Google Drive copy containing IMAGE and MASK folders or configure Kaggle API credentials for "
        "sagnikkayalcse52/casia-spicing-detection-localization."
    )

print("Normalized dataset root:", dataset_dir)
print("IMAGE dir exists:", (dataset_dir / "IMAGE").exists())
print("MASK dir exists:", (dataset_dir / "MASK").exists())

Google Drive mount/search skipped: No module named 'google.colab'


You must authenticate before you can call the Kaggle API.
Follow the instructions to authenticate at: https://github.com/Kaggle/kaggle-cli/blob/main/docs/README.md#authentication
Kaggle API import failed: name 'exit' is not defined


FileNotFoundError: Could not prepare /kaggle/input/casia-spicing-detection-localization. Provide a Google Drive copy containing IMAGE and MASK folders or configure Kaggle API credentials for sagnikkayalcse52/casia-spicing-detection-localization.

### 1.4 Suppress libpng Warnings

Redirects C-level stderr to `/dev/null` so that repetitive libpng decoding warnings
do not clutter notebook output during image loading.

In [ ]:
import os
devnull = os.open(os.devnull, os.O_WRONLY)
os.dup2(devnull, 2)  # redirect C-level stderr to /dev/null


## 2. Dataset Explanation

The notebook operates on an image tampering dataset organized into authentic and tampered image folders with paired binary masks.

- **Authentic images** represent unmodified examples.
- **Tampered images** represent forged examples.
- **Masks** are binary supervision targets where manipulated pixels are foreground and clean pixels are background.
- **Splitting strategy** uses a `70 / 15 / 15` train-validation-test split produced from the metadata CSVs created below.

**Section breakdown**
- discover IMAGE and MASK directories
- build a metadata table of image-mask pairs
- split metadata into train, validation, and test subsets

**Assignment alignment:** This section prepares the paired supervision needed for both tamper detection and tampered-region localization.

### 2.1 Discover Dataset Root and Build Metadata

This subsection inspects the Kaggle-style input directory to confirm the dataset is present,
then iterates over `IMAGE/Au`, `IMAGE/Tp`, `MASK/Au`, and `MASK/Tp` subdirectories
to build a metadata table of image-mask pairs.

#### 2.1.1 Locate IMAGE and MASK Directories

Scans the normalized Kaggle-style input path and searches recursively for the `IMAGE` and `MASK` folders.

In [ ]:
import os
from pathlib import Path
import pandas as pd

# =========================
# 1) Discover the dataset root
# =========================

# Inspect the Kaggle-style input directory so the notebook can confirm the expected dataset is available.
INPUT_ROOT = Path("/kaggle/input")

print("Available datasets under /kaggle/input:")
for p in INPUT_ROOT.iterdir():
    if p.is_dir():
        print(" -", p.name)

# Keep the original path expression intact because later cells assume this exact dataset location.
DATASET_DIR = INPUT_ROOT / "/kaggle/input/casia-spicing-detection-localization"

# Search recursively for IMAGE and MASK so the metadata build still works if the dataset is nested one level deeper.
IMAGE_DIR = None
MASK_DIR = None

for p in DATASET_DIR.rglob("*"):
    if p.is_dir() and p.name.lower() == "image":
        IMAGE_DIR = p
    if p.is_dir() and p.name.lower() == "mask":
        MASK_DIR = p

print("IMAGE_DIR:", IMAGE_DIR)
print("MASK_DIR:", MASK_DIR)

assert IMAGE_DIR is not None, "Could not find the IMAGE directory. Verify the dataset folder name and structure."
assert MASK_DIR is not None, "Could not find the MASK directory. Verify the dataset folder name and structure."


#### 2.1.2 Build Metadata Table from Image-Mask Pairs

Iterates over authentic (`Au`) and tampered (`Tp`) subdirectories,
pairing each image with its corresponding mask and recording the image-level label.

In [ ]:
# =========================
# 2) Build metadata for images and masks
# =========================

# The source dataset is expected to follow the standard CASIA naming convention:
#   IMAGE/Au -> authentic source images
#   IMAGE/Tp -> tampered images
#   MASK/Au  -> masks for authentic images
#   MASK/Tp  -> masks for tampered images
label_folders = {
    "Au": {"class_name": "authentic", "label": 0},
    "Tp": {"class_name": "tampered",  "label": 1},
}

valid_exts = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}

rows = []

for sub_name, info in label_folders.items():
    img_subdir = IMAGE_DIR / sub_name
    mask_subdir = MASK_DIR / sub_name

    if not img_subdir.exists():
        print(f"Warning: image folder does not exist: {img_subdir}")
        continue

    print(f"Scanning images in: {img_subdir}")

    # Iterate over filenames only so metadata generation stays memory-safe for large datasets.
    for img_path in img_subdir.iterdir():
        if not img_path.is_file():
            continue

        if img_path.suffix.lower() not in valid_exts:
            continue

        # Assume the mask uses the same filename inside the matching MASK subdirectory.
        mask_path = mask_subdir / img_path.name
        mask_exists = mask_path.exists()

        rows.append({
            "image_path": str(img_path),
            "mask_path": str(mask_path) if mask_exists else "",
            "mask_exists": int(mask_exists),
            "class_folder": sub_name,
            "class_name": info["class_name"],
            "label": info["label"],
        })

print(f"Total images found: {len(rows)}")


#### 2.1.3 Save Metadata CSV

Persists the image-mask-label metadata to a CSV file for use by downstream data loading cells.

In [ ]:
# =========================
# 3) Save the metadata CSV
# =========================

df = pd.DataFrame(rows)
print(df.head())

output_csv = "/kaggle/working/image_mask_metadata.csv"
df.to_csv(output_csv, index=False)
print("CSV saved to:", output_csv)

# Print a small sanity check so the class balance and missing-mask cases are visible before training starts.
print("\nCounts per class_name:")
print(df["class_name"].value_counts())

print("\nMissing masks:")
print(df[df["mask_exists"] == 0].head())

### 2.2 Train / Validation / Test Split

Loads the metadata CSV, applies a stratified 70/15/15 split to preserve class balance
across train, validation, and test subsets, and saves each split to a separate CSV file.

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from pathlib import Path

# 1) Load the metadata CSV generated in the previous step.
csv_path = Path("/kaggle/working/image_mask_metadata.csv")
df = pd.read_csv(csv_path)

print("Total samples:", len(df))
print(df["class_name"].value_counts())

# 2) Split the dataset into train, validation, and test subsets using stratified sampling.
# Stratification preserves the authentic vs tampered ratio across all three splits.
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42,
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42,
)

print("Train size:", len(train_df))
print("Val size:  ", len(val_df))
print("Test size: ", len(test_df))

print("\nTrain class distribution:")
print(train_df["class_name"].value_counts())

print("\nVal class distribution:")
print(val_df["class_name"].value_counts())

print("\nTest class distribution:")
print(test_df["class_name"].value_counts())

# 3) Save the split metadata files so the downstream training code can load them directly.
output_dir = Path("/kaggle/working")
output_dir.mkdir(parents=True, exist_ok=True)

train_csv = output_dir / "train_metadata.csv"
val_csv   = output_dir / "val_metadata.csv"
test_csv  = output_dir / "test_metadata.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv, index=False)
test_df.to_csv(test_csv, index=False)

print("\nSaved:")
print(" -", train_csv)
print(" -", val_csv)
print(" -", test_csv)

## 3. Data Loading and Preprocessing

The source notebook loads metadata CSV files, builds PyTorch datasets and dataloaders, resizes inputs to `256 x 256`, normalizes RGB channels, and applies augmentation for the training split.

This notebook preserves the original preprocessing logic exactly and only adds documentation around it.

**Assignment alignment:** These preprocessing steps prepare inputs for both image-level tamper classification and pixel-level mask prediction.

## Source-Preserved Prior Experiment Block

The next code cell is retained from the source notebook for provenance. It contains an earlier training configuration and is documented here so readers can understand how the pipeline evolved.

**Subsections covered in the block**
- warning suppression and imports
- metadata loading
- augmentations and dataset behavior
- U-Net plus classifier architecture
- dataloader setup
- training and validation routines

**Assignment alignment:** Even though this is not the final submission run, it still demonstrates the same assignment objective: joint tamper detection and tampered-region localization in a single notebook pipeline.

#### 3.1 Package Installation

Installs `albumentations` and `opencv-python-headless` for image augmentation and loading.

In [ ]:
!pip install -q albumentations==1.3.1 opencv-python-headless==4.10.0.84

#### 3.2 Warning Suppression and Imports

Defines a context manager that temporarily silences stderr to suppress repetitive
libpng/OpenCV warnings during image loading, followed by all required imports.

In [ ]:

import sys, os, contextlib

# =====================================================
# 1) Suppress image-reading warnings from cv2 and PIL
# =====================================================

@contextlib.contextmanager
def suppress_image_warnings():
    """
    Purpose:
        Temporarily silence stderr while reading images and masks.

    Inputs:
        None.

    Returns:
        contextmanager: A context that suppresses noisy image-decoding warnings.

    Notes:
        This is used to hide repetitive libpng warnings without changing the image-loading logic itself.
    """
    devnull = open(os.devnull, "w")
    old_stderr = sys.stderr
    sys.stderr = devnull
    try:
        yield
    finally:
        sys.stderr = old_stderr
        devnull.close()

# =====================================================
# 2) Imports
# =====================================================

import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2


#### 3.3 Metadata Loading

Reads the train, validation, and test CSV files produced by the dataset preparation step.

In [ ]:
# =====================================================
# 3) Load metadata CSV files
# =====================================================

TRAIN_CSV = "/kaggle/working/test_metadata.csv"
VAL_CSV   = "/kaggle/working/val_metadata.csv"
TEST_CSV  = "/kaggle/working/val_metadata.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print(train_df["class_name"].value_counts())


#### 3.4 Image and Mask Transforms

Defines Albumentations pipelines for the training split (with augmentation) and the
validation/test splits (resize and normalize only).

In [ ]:
# =====================================================
# 4) Define image and mask transforms
# =====================================================

IMAGE_SIZE = 256

def get_train_transform():
    """
    Purpose:
        Build the augmentation pipeline used for the training split.

    Inputs:
        None.

    Returns:
        A.Compose: Albumentations pipeline for training images and masks.

    Notes:
        The transform resizes inputs, applies light geometric and photometric augmentation, normalizes RGB channels,
        and converts tensors for PyTorch training.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.ShiftScaleRotate(
            shift_limit=0.02,
            scale_limit=0.1,
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5,
        ),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_valid_transform():
    """
    Purpose:
        Build the deterministic preprocessing pipeline for validation and test data.

    Inputs:
        None.

    Returns:
        A.Compose: Albumentations pipeline without stochastic augmentation.

    Notes:
        Validation and test examples are only resized and normalized so evaluation remains stable across runs.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


#### 3.5 Dataset Class Definition

The `ImageMaskDataset` class reads paired images and masks, applies transforms,
and returns tensors suitable for classification and segmentation training.

In [ ]:
# =====================================================
# 5) Dataset definition
# =====================================================

class ImageMaskDataset(Dataset):
    """
    Purpose:
        Load paired images, binary masks, and image-level tamper labels from the metadata table.

    Key Responsibilities:
        - Read each image-mask pair from disk.
        - Convert masks into binary localization targets and apply shared transforms.

    Notes:
        The dataset returns a tuple of image tensor, mask tensor, and image-level class label.
    """
    def __init__(self, df, transform=None):
        """
        Purpose:
            Store metadata and an optional augmentation pipeline for later sample retrieval.

        Inputs:
            df (pd.DataFrame): Metadata table containing image paths, mask paths, and labels.
            transform (callable): Optional Albumentations pipeline applied to both image and mask.

        Returns:
            None.

        Notes:
            The dataframe index is reset so __getitem__ can rely on stable integer indexing.
        """
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        """
        Purpose:
            Return the number of examples available in the dataset.

        Inputs:
            None.

        Returns:
            int: Number of rows in the metadata table.

        Notes:
            This length is used by PyTorch DataLoader to determine epoch size.
        """
        return len(self.df)

    def __getitem__(self, idx):
        """
        Purpose:
            Load one image, its binary mask, and the corresponding image-level tamper label.

        Inputs:
            idx (int): Row index inside the metadata table.

        Returns:
            tuple: Image tensor, mask tensor, and class label tensor.

        Notes:
            Masks are binarized so the segmentation branch learns a clean tampered-vs-clean boundary.
        """
        row = self.df.iloc[idx]

        img_path = row["image_path"]
        mask_path = row["mask_path"]
        label = int(row["label"])

        # Read image and mask while suppressing repetitive backend warnings.
        with suppress_image_warnings():
            image = cv2.imread(img_path)
            mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Failed to read image: {img_path}")
        if mask is None:
            raise RuntimeError(f"Failed to read mask: {mask_path}")

        # Convert BGR to RGB for visualization and normalization consistency.
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Convert mask values to binary so the model predicts tampered vs non-tampered pixels.
        mask = (mask > 0).astype("float32")

        with suppress_image_warnings():
            if self.transform is not None:
                # Apply the same spatial transform to the image and its mask.
                augmented = self.transform(image=image, mask=mask)
                image = augmented["image"]
                mask = augmented["mask"]
            else:
                image = torch.from_numpy(image).permute(2,0,1).float() / 255.0
                mask = torch.from_numpy(mask).unsqueeze(0).float()

        if mask.ndim == 2:
            # Ensure the mask keeps a channel dimension for BCE-style segmentation losses.
            mask = mask.unsqueeze(0)

        return image, mask, torch.tensor(label, dtype=torch.long)


#### 3.6 U-Net with Classification Head

Defines the `DoubleConv`, `Down`, `Up`, and `UNetWithClassifier` modules that form
the shared encoder-decoder backbone and the bottleneck classification head.

In [ ]:
# =====================================================
# 6) Define the U-Net with a classification head
# =====================================================

class DoubleConv(nn.Module):
    """
    Purpose:
        Apply two consecutive convolution blocks used throughout the encoder and decoder.

    Key Responsibilities:
        - Extract local features while preserving spatial resolution.
        - Provide a reusable building block for the U-Net backbone.

    Notes:
        Each block uses Conv-BatchNorm-ReLU twice.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Construct the two-layer convolutional block.

        Inputs:
            in_ch (int): Number of input feature channels.
            out_ch (int): Number of output feature channels.

        Returns:
            None.

        Notes:
            Bias is disabled because batch normalization follows each convolution.
        """
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        """
        Purpose:
            Run the input tensor through the double-convolution feature extractor.

        Inputs:
            x (torch.Tensor): Input feature map.

        Returns:
            torch.Tensor: Refined feature map with updated channels.

        Notes:
            This helper is shared by both encoder and decoder stages.
        """
        return self.block(x)

class Down(nn.Module):
    """
    Purpose:
        Perform one encoder-stage downsampling step in the U-Net backbone.

    Key Responsibilities:
        - Reduce spatial resolution with max pooling.
        - Increase representational capacity with a DoubleConv block.

    Notes:
        This module is used to move from shallow texture features to deeper semantic features.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Initialize the pooling and convolutional layers for one encoder stage.

        Inputs:
            in_ch (int): Number of input feature channels.
            out_ch (int): Number of output feature channels.

        Returns:
            None.

        Notes:
            Pooling is applied before the DoubleConv block.
        """
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        """
        Purpose:
            Downsample the incoming feature map and refine it with convolution.

        Inputs:
            x (torch.Tensor): Input feature map.

        Returns:
            torch.Tensor: Downsampled feature map for the next encoder stage.

        Notes:
            Spatial dimensions are halved before convolution.
        """
        return self.conv(self.pool(x))

class Up(nn.Module):
    """
    Purpose:
        Perform one decoder-stage upsampling step and merge encoder skip features.

    Key Responsibilities:
        - Upsample coarse decoder features.
        - Concatenate them with the corresponding encoder activations.

    Notes:
        Padding is applied when feature-map sizes differ by one pixel after transposed convolution.
    """
    def __init__(self, in_channels, out_channels):
        """
        Purpose:
            Initialize one U-Net decoder block.

        Inputs:
            in_channels (int): Number of channels entering the decoder block.
            out_channels (int): Number of channels after upsampling.

        Returns:
            None.

        Notes:
            The concatenated tensor is passed through DoubleConv for feature fusion.
        """
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        """
        Purpose:
            Upsample decoder features and fuse them with the matching encoder skip connection.

        Inputs:
            x1 (torch.Tensor): Decoder feature map from the deeper stage.
            x2 (torch.Tensor): Encoder feature map used as a skip connection.

        Returns:
            torch.Tensor: Fused decoder feature map.

        Notes:
            Padding keeps tensor shapes compatible before concatenation.
        """
        x1 = self.up(x1)

        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX//2, diffX-diffX//2,
                        diffY//2, diffY-diffY//2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNetWithClassifier(nn.Module):
    """
    Purpose:
        Predict both image-level tamper labels and pixel-level tampered regions in one forward pass.

    Key Responsibilities:
        - Produce a segmentation logit map for localization.
        - Produce classification logits from the bottleneck for image-level detection.

    Notes:
        The encoder is shared between both tasks so classification and localization learn from the same features.
    """
    def __init__(self, n_channels=3, n_classes=1, n_labels=2):
        """
        Purpose:
            Build the full encoder-decoder network and the bottleneck classifier head.

        Inputs:
            n_channels (int): Number of input image channels.
            n_classes (int): Number of segmentation output channels.
            n_labels (int): Number of image-level classes.

        Returns:
            None.

        Notes:
            The architecture is preserved exactly from the source notebook.
        """
        super().__init__()

        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)

        self.outc = nn.Conv2d(64, n_classes, 1)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1,1)),
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_labels)
        )

    def forward(self, x):
        """
        Purpose:
            Run the shared encoder-decoder and return both classification and segmentation outputs.

        Inputs:
            x (torch.Tensor): Batch of input images.

        Returns:
            tuple: Image-level classification logits and pixel-level segmentation logits.

        Notes:
            The classification head operates on the bottleneck feature map, while the decoder produces the mask logits.
        """
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)

        seg_logits = self.outc(x)
        cls_logits = self.classifier(x5)

        return cls_logits, seg_logits


#### 3.7 DataLoader Construction

Creates PyTorch `DataLoader` instances for the train, validation, and test datasets.

In [ ]:
# =====================================================
# 7) Build dataloaders
# =====================================================

BATCH_SIZE = 8
NUM_WORKERS = 2

train_dataset = ImageMaskDataset(train_df, transform=get_train_transform())
val_dataset   = ImageMaskDataset(val_df,   transform=get_valid_transform())
test_dataset  = ImageMaskDataset(test_df,  transform=get_valid_transform())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)


#### 3.8 Training Loop and Validation

Runs the prior experiment configuration: trains for 30 epochs using Adam with ReduceLROnPlateau,
checkpoints the best model by validation accuracy, and evaluates on the test split.

In [ ]:
# =====================================================
# 8) Train the earlier experiment configuration
# =====================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = UNetWithClassifier().to(device)

class_weights = compute_class_weight("balanced", classes=np.array([0, 1]), y=train_df["label"].values)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

criterion_cls = nn.CrossEntropyLoss(weight=class_weights)
criterion_seg = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=3)

ALPHA = 1.0
BETA  = 1.0

def dice_coef(preds, targets, eps=1e-7):
    """
    Purpose:
        Compute the Dice coefficient for thresholded segmentation predictions.

    Inputs:
        preds (torch.Tensor): Raw segmentation logits from the model.
        targets (torch.Tensor): Ground-truth binary masks.
        eps (float): Small constant for numerical stability.

    Returns:
        float: Mean Dice score across the batch.

    Notes:
        Dice emphasizes overlap quality, which is useful for tampered-region localization.
    """
    probs = torch.sigmoid(preds)
    preds_bin = (probs > 0.5).float()
    intersection = (preds_bin * targets).sum((1,2,3))
    union = preds_bin.sum((1,2,3)) + targets.sum((1,2,3))
    return ((2.0*intersection + eps) / (union + eps)).mean().item()

def train_one_epoch(epoch):
    """
    Purpose:
        Train the model for one epoch on the training split.

    Inputs:
        epoch (int): Current epoch number for logging.

    Returns:
        None.

    Notes:
        This earlier experiment combines image-level classification loss with segmentation loss.
    """
    model.train()
    total_loss = 0
    correct = 0

    for images, masks, labels in train_loader:
        images, masks, labels = images.to(device), masks.to(device), labels.to(device)

        # Reset gradients before the next optimization step.
        optimizer.zero_grad()

        # Forward pass: produce image-level logits and pixel-level mask logits.
        cls_logits, seg_logits = model(images)

        # Combine classification and segmentation objectives using the preserved task weights.
        loss = ALPHA*criterion_cls(cls_logits, labels) + BETA*criterion_seg(seg_logits, masks)

        # Backpropagation: compute gradients for the shared backbone and both task heads.
        loss.backward()
        # Optimizer step: update model parameters.
        optimizer.step()

        total_loss += loss.item() * images.size(0)
        correct += (torch.argmax(cls_logits, dim=1) == labels).sum().item()

    print(f"Epoch {epoch} | Train Loss: {total_loss/len(train_dataset):.4f} | Train Acc: {correct/len(train_dataset):.4f}")

def validate(epoch, loader, name="Val"):
    """
    Purpose:
        Evaluate the model on a validation-style dataloader.

    Inputs:
        epoch (int | str): Epoch identifier for logging.
        loader (DataLoader): Validation or test dataloader.
        name (str): Split name used in printed summaries.

    Returns:
        float: Image-level classification accuracy for the evaluated split.

    Notes:
        Dice is reported to summarize mask quality without affecting the checkpoint rule in this block.
    """
    model.eval()
    total_loss = 0
    correct = 0
    dice_sum = 0
    count = 0

    with torch.no_grad():
        for images, masks, labels in loader:
            images, masks, labels = images.to(device), masks.to(device), labels.to(device)

            # Forward pass: reuse the shared model for evaluation without gradient tracking.
            cls_logits, seg_logits = model(images)

            # Compute the same multi-task objective used during training for consistent reporting.
            loss = ALPHA*criterion_cls(cls_logits, labels) + BETA*criterion_seg(seg_logits, masks)

            total_loss += loss.item() * images.size(0)
            correct += (torch.argmax(cls_logits, dim=1) == labels).sum().item()

            # Convert segmentation output into a thresholded overlap metric for localization quality.
            dice_sum += dice_coef(seg_logits, masks)
            count += 1

    print(f"Epoch {epoch} | {name} Loss: {total_loss/len(loader.dataset):.4f} | {name} Acc: {correct/len(loader.dataset):.4f} | {name} Dice: {dice_sum/count:.4f}")

    return correct/len(loader.dataset)

NUM_EPOCHS = 30
best_acc = 0
best_model_path = "/kaggle/working/best_model.pth"

for epoch in range(1, NUM_EPOCHS+1):
    train_one_epoch(epoch)
    val_acc = validate(epoch, val_loader, "Val")

    scheduler.step(val_acc)

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), best_model_path)
        print("==> Saved best model (val acc:", best_acc, ")")

print("Training finished. Best Val Acc:", best_acc)

model.load_state_dict(torch.load(best_model_path))
test_acc = validate("Final", test_loader, "Test")

## 4. Model Architecture

The implemented model is a custom U-Net-style encoder-decoder with a classifier head.

- `DoubleConv`, `Down`, and `Up` blocks define the encoder-decoder backbone.
- A segmentation head outputs a one-channel tampering mask.
- A separate classification head attached to the bottleneck predicts whether the image is authentic or tampered.

**Assignment alignment:** This shared architecture directly supports the two required tasks: image-level tamper detection and pixel-level tampered-region localization.

## 5. Training Strategy

The effective submission run uses:

- **Optimizer:** `Adam`
- **Learning rate:** `1e-4`
- **Batch size:** `8`
- **Epochs:** `50`
- **Scheduler:** `CosineAnnealingLR(T_max=10)`
- **Classification loss:** focal-style cross-entropy
- **Segmentation loss:** equal mixture of BCE-with-logits and Dice loss

Importantly, these values are preserved from the source notebook. The additions below only improve reporting and assignment presentation.

**Assignment alignment:** This section shows how the model is optimized to learn both image-level tamper detection and mask-level localization jointly.

## 6. Hyperparameters

| Hyperparameter | Effective Value |
|---|---|
| `IMAGE_SIZE` | `256` |
| `BATCH_SIZE` | `8` |
| `NUM_WORKERS` | `2` |
| `lr` | `1e-4` |
| `NUM_EPOCHS` | `50` |
| `ALPHA` | `1.5` |
| `BETA` | `1.0` |
| `scheduler` | `CosineAnnealingLR(T_max=10)` |
| classifier dropout | `0.5` |
| segmentation threshold for reporting | `0.5` |

**Assignment alignment:** These preserved settings define the exact training conditions used to produce the submission results.

## 7. Experiment Tracking

W&B is attached only to the effective submission run below.

- The run is enabled by default.
- If online authentication is unavailable, the notebook falls back to offline logging instead of breaking execution.
- Logged metrics include training loss, validation loss, accuracy, Dice, IoU, and F1.
- Qualitative prediction grids and training curves are also logged when available.

**Assignment alignment:** Experiment tracking improves reproducibility and makes it easier to report the training and evaluation evidence expected in the assignment submission.

In [ ]:
import importlib.util
import subprocess
import sys

WANDB_ACTIVE = False
WANDB_RUN = None
WANDB_MODE = "disabled"
WANDB_CONFIG = {
    "architecture": "UNetWithClassifier",
    "image_size": 256,
    "batch_size": 8,
    "num_workers": 2,
    "learning_rate": 1e-4,
    "epochs": 50,
    "classification_loss_weight": 1.5,
    "segmentation_loss_weight": 1.0,
    "scheduler": "CosineAnnealingLR(T_max=10)",
    "dropout": 0.5,
    "dataset": "CASIA tampered vs authentic with binary masks",
    "runtime": "Google Colab",
}

try:
    # Install W&B only when it is missing so the notebook stays self-contained in a fresh Colab runtime.
    if importlib.util.find_spec("wandb") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "wandb"])

    import wandb

    try:
        # Prefer online logging for the final submission narrative when authentication is available.
        wandb.login()
        WANDB_RUN = wandb.init(
            project="tampered-image-detection-assignment",
            name="vk3-unetwithclassifier-colab",
            tags=["assignment", "colab", "tampering", "localization"],
            config=WANDB_CONFIG,
            reinit=True,
        )
        WANDB_MODE = "online"
    except Exception as auth_exc:
        # Fall back to offline logging so experiment tracking never blocks notebook execution.
        print(f"W&B online login unavailable, switching to offline mode: {auth_exc}")
        WANDB_RUN = wandb.init(
            project="vK.7 Image Detection and Localisation.ipynb",
            name="vk7-unetwithclassifier-colab-offline",
            tags=["assignment", "colab", "tampering", "localization", "offline"],
            config=WANDB_CONFIG,
            mode="offline",
            reinit=True,
        )
        WANDB_MODE = "offline"

    WANDB_ACTIVE = WANDB_RUN is not None
except Exception as wandb_exc:
    WANDB_ACTIVE = False
    WANDB_RUN = None
    print(f"W&B setup unavailable; continuing without active logging: {wandb_exc}")

print("W&B active:", WANDB_ACTIVE)
print("W&B mode:", WANDB_MODE)

## 8. Effective Submission Training Loop

The next code cell is the effective final training and evaluation block used for the submission narrative.
The underlying model, optimizer, scheduler, losses, and checkpoint rule are preserved.
Only reporting additions were introduced: W&B logging and derived IoU/F1 metrics that do not change training decisions.

**Subsections covered in the block**
- metadata loading
- transforms and dataset behavior
- model definition
- loss functions and reporting metrics
- training loop and checkpointing
- validation and final test evaluation

**Assignment alignment:** This is the main assignment run that produces both image-level detection performance and mask-level localization performance.

### 8.1 Dependencies and Imports

Installs required packages, silences libpng warnings, and imports all libraries
needed for the effective submission run.

In [ ]:
# ================== Install dependencies and silence libpng warnings ==================
import os
devnull = os.open(os.devnull, os.O_WRONLY)
os.dup2(devnull, 2)   # Redirect stderr so repeated libpng warnings do not flood notebook output.

!pip install -q albumentations==1.3.1 opencv-python-headless==4.10.0.84

# ================== Imports ==================
import cv2
import sys
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

WANDB_ACTIVE = globals().get("WANDB_ACTIVE", False)
WANDB_RUN = globals().get("WANDB_RUN", None)


### 8.2 Metadata Loading

Reads the train, validation, and test metadata CSV files generated during dataset preparation.

In [ ]:
# ================== 1) Load metadata files ==================
# Keep the original file locations intact so the documented notebook matches the source pipeline.
TRAIN_CSV = "/kaggle/working/train_metadata.csv"
VAL_CSV   = "/kaggle/working/val_metadata.csv"
TEST_CSV  = "/kaggle/working/test_metadata.csv"

train_df = pd.read_csv(TRAIN_CSV)
val_df   = pd.read_csv(VAL_CSV)
test_df  = pd.read_csv(TEST_CSV)

print("Train:", len(train_df), "Val:", len(val_df), "Test:", len(test_df))
print("Train class distribution:\n", train_df["class_name"].value_counts())


### 8.3 Image and Mask Transforms

Defines the Albumentations augmentation pipeline for training (with flip, brightness,
noise, JPEG compression, and shift-scale-rotate) and the deterministic pipeline for validation/test.

In [ ]:
# ================== 2) Define transforms ==================
IMAGE_SIZE = 256

def get_train_transform():
    """
    Purpose:
        Build the augmentation pipeline used for training images and masks.

    Inputs:
        None.

    Returns:
        A.Compose: Albumentations pipeline for the training split.

    Notes:
        The transform combines resizing, light augmentation, normalization, and tensor conversion.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.HorizontalFlip(p=0.5),
        A.RandomBrightnessContrast(p=0.3),
        A.GaussNoise(p=0.25),
        A.JpegCompression(quality_lower=50, quality_upper=90, p=0.25),
        A.ShiftScaleRotate(
            shift_limit=0.02,
            scale_limit=0.1,
            rotate_limit=10,
            border_mode=cv2.BORDER_REFLECT_101,
            p=0.5,
        ),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])

def get_valid_transform():
    """
    Purpose:
        Build the deterministic preprocessing pipeline for validation and test samples.

    Inputs:
        None.

    Returns:
        A.Compose: Albumentations pipeline without stochastic augmentation.

    Notes:
        Using the same resize and normalization scheme keeps evaluation comparable to training inputs.
    """
    return A.Compose([
        A.Resize(IMAGE_SIZE, IMAGE_SIZE),
        A.Normalize(mean=(0.485, 0.456, 0.406),
                    std=(0.229, 0.224, 0.225)),
        ToTensorV2(),
    ])


### 8.4 Dataset Class

The `ImageMaskDataset` class loads image-mask pairs from metadata, applies shared transforms,
and returns tensors suitable for joint classification and segmentation training.

In [ ]:
# ================== 3) Define the dataset ==================
class ImageMaskDataset(Dataset):
    """
    Purpose:
        Load paired images, masks, and image-level tamper labels from the metadata CSV files.

    Key Responsibilities:
        - Read each image and its corresponding mask from disk.
        - Apply shared preprocessing so classification and localization are trained on aligned inputs.

    Notes:
        The dataset returns image tensor, binary mask tensor, and image-level label.
    """
    def __init__(self, df, transform=None):
        """
        Purpose:
            Store metadata and an optional transform pipeline for later sample loading.

        Inputs:
            df (pd.DataFrame): Metadata table with image paths, mask paths, and labels.
            transform (callable): Optional Albumentations transform applied to image and mask.

        Returns:
            None.

        Notes:
            The dataframe index is reset to preserve stable integer access during training.
        """
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        """
        Purpose:
            Return the number of examples in the dataset.

        Inputs:
            None.

        Returns:
            int: Number of metadata rows.

        Notes:
            DataLoader uses this length to define epoch size.
        """
        return len(self.df)

    def __getitem__(self, idx):
        """
        Purpose:
            Load one image, its binary mask, and the image-level tamper label.

        Inputs:
            idx (int): Sample index inside the metadata table.

        Returns:
            tuple: Image tensor, mask tensor, and class label tensor.

        Notes:
            Binary masks make the segmentation target explicit: tampered pixels vs clean pixels.
        """
        row = self.df.iloc[idx]

        img_path = row["image_path"]
        mask_path = row["mask_path"]
        label = int(row["label"])

        image = cv2.imread(img_path)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        if image is None:
            raise RuntimeError(f"Failed to read image: {img_path}")
        if mask is None:
            raise RuntimeError(f"Failed to read mask: {mask_path}")

        # Convert the OpenCV image to RGB before normalization and visualization.
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        # Convert the grayscale mask to binary so the localization target has a clear foreground/background split.
        mask = (mask > 0).astype("float32")

        if self.transform is not None:
            # Apply identical spatial augmentation to the image and its supervision mask.
            augmented = self.transform(image=image, mask=mask)
            image = augmented["image"]
            mask  = augmented["mask"]
        else:
            image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
            mask  = torch.from_numpy(mask).unsqueeze(0).float()

        if mask.ndim == 2:
            # Keep an explicit channel dimension for the segmentation branch and BCE-style losses.
            mask = mask.unsqueeze(0)

        return image, mask, torch.tensor(label, dtype=torch.long)


### 8.5 U-Net with Classifier Architecture

Defines the encoder-decoder backbone (`DoubleConv`, `Down`, `Up`) and the full
`UNetWithClassifier` model that produces both image-level classification logits
and pixel-level segmentation logits from a shared representation.

In [ ]:
# ================== 4) Define the U-Net with classifier head ==================
class DoubleConv(nn.Module):
    """
    Purpose:
        Apply two convolutional layers with normalization and activation.

    Key Responsibilities:
        - Extract localized visual features.
        - Reuse the same feature block across encoder and decoder stages.

    Notes:
        This block is the core building unit of the U-Net backbone.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Initialize the double-convolution block.

        Inputs:
            in_ch (int): Number of input channels.
            out_ch (int): Number of output channels.

        Returns:
            None.

        Notes:
            Batch normalization follows each convolution.
        """
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        """
        Purpose:
            Transform the input feature map with two convolution stages.

        Inputs:
            x (torch.Tensor): Input feature tensor.

        Returns:
            torch.Tensor: Feature tensor after convolution, normalization, and activation.

        Notes:
            This helper keeps encoder and decoder definitions concise.
        """
        return self.block(x)

class Down(nn.Module):
    """
    Purpose:
        Downsample feature maps in the encoder path.

    Key Responsibilities:
        - Reduce spatial resolution with max pooling.
        - Expand channel capacity with a DoubleConv block.

    Notes:
        This block helps the network capture higher-level tampering cues.
    """
    def __init__(self, in_ch, out_ch):
        """
        Purpose:
            Initialize one encoder stage.

        Inputs:
            in_ch (int): Number of incoming channels.
            out_ch (int): Number of outgoing channels.

        Returns:
            None.

        Notes:
            Pooling is applied before the convolution block.
        """
        super().__init__()
        self.pool = nn.MaxPool2d(2)
        self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x):
        """
        Purpose:
            Downsample the input feature map and refine it.

        Inputs:
            x (torch.Tensor): Encoder input feature map.

        Returns:
            torch.Tensor: Downsampled and refined feature map.

        Notes:
            The sequence preserves the original implementation exactly.
        """
        x = self.pool(x)
        x = self.conv(x)
        return x

class Up(nn.Module):
    """
    Purpose:
        Upsample decoder features and fuse them with encoder skip connections.

    Key Responsibilities:
        - Recover spatial detail in the decoder.
        - Merge coarse semantic features with fine-grained encoder activations.

    Notes:
        Padding compensates for minor shape mismatches before concatenation.
    """
    def __init__(self, in_channels, out_channels):
        """
        Purpose:
            Initialize one decoder stage.

        Inputs:
            in_channels (int): Number of channels entering the transposed convolution.
            out_channels (int): Number of channels produced after upsampling.

        Returns:
            None.

        Notes:
            The concatenated tensor is refined with DoubleConv.
        """
        super().__init__()
        self.up = nn.ConvTranspose2d(in_channels, out_channels, 2, stride=2)
        self.conv = DoubleConv(in_channels, out_channels)

    def forward(self, x1, x2):
        """
        Purpose:
            Upsample decoder features and concatenate them with encoder skip features.

        Inputs:
            x1 (torch.Tensor): Decoder feature map from the deeper level.
            x2 (torch.Tensor): Encoder skip connection from the matching spatial scale.

        Returns:
            torch.Tensor: Decoder feature map after skip fusion.

        Notes:
            Shape alignment is handled with explicit padding before concatenation.
        """
        x1 = self.up(x1)

        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]

        x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
                        diffY // 2, diffY - diffY // 2])

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class UNetWithClassifier(nn.Module):
    """
    Purpose:
        Predict both image-level tampering labels and pixel-level tampering masks.

    Key Responsibilities:
        - Use a shared encoder-decoder backbone for localization.
        - Use a classifier head on the bottleneck to detect whether an image is tampered.

    Notes:
        The shared backbone lets the model learn complementary detection and localization signals.
    """
    def __init__(self, n_channels=3, n_classes=1, n_labels=2):
        """
        Purpose:
            Construct the shared U-Net backbone and the auxiliary classifier head.

        Inputs:
            n_channels (int): Number of image channels.
            n_classes (int): Number of segmentation output channels.
            n_labels (int): Number of classification labels.

        Returns:
            None.

        Notes:
            The architecture is preserved from the original notebook without structural changes.
        """
        super().__init__()

        # Encoder
        self.inc   = DoubleConv(n_channels, 64)
        self.down1 = Down(64, 128)
        self.down2 = Down(128, 256)
        self.down3 = Down(256, 512)
        self.down4 = Down(512, 1024)

        # Decoder
        self.up1 = Up(1024, 512)
        self.up2 = Up(512, 256)
        self.up3 = Up(256, 128)
        self.up4 = Up(128, 64)

        self.outc = nn.Conv2d(64, n_classes, kernel_size=1)

        # Classification head
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten(),
            nn.Linear(1024, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(512, n_labels),
        )

    def forward(self, x):
        """
        Purpose:
            Produce image-level logits and segmentation logits from the same input batch.

        Inputs:
            x (torch.Tensor): Batch of input images.

        Returns:
            tuple: Classification logits and segmentation logits.

        Notes:
            Classification uses the bottleneck representation, while segmentation uses the decoder output.
        """
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)   # bottleneck

        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)
        seg_logits = self.outc(x)

        cls_logits = self.classifier(x5)

        return cls_logits, seg_logits


### 8.6 DataLoader Construction

Creates train, validation, and test `DataLoader` instances with batch size 8 and 2 workers.

In [ ]:
# ================== 5) Build dataloaders ==================
BATCH_SIZE = 8
NUM_WORKERS = 2

train_dataset = ImageMaskDataset(train_df, transform=get_train_transform())
val_dataset   = ImageMaskDataset(val_df,   transform=get_valid_transform())
test_dataset  = ImageMaskDataset(test_df,  transform=get_valid_transform())

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=True)


### 8.7 Loss Functions and Reporting Metrics

Defines the focal-style classification loss, the combined BCE + Dice segmentation loss,
and the Dice coefficient used for checkpoint-independent reporting.
Configures the Adam optimizer with learning rate `1e-4` and CosineAnnealingLR scheduler.

In [ ]:
# ================== 6) Loss functions and reporting metrics ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

model = UNetWithClassifier().to(device)

# Compute class weights from the training split so the image-level classifier handles class imbalance more fairly.
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].values,
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
print("Class weights:", class_weights)

class FocalLoss(nn.Module):
    """
    Purpose:
        Apply focal-style weighting to the image-level classification loss.

    Key Responsibilities:
        - Down-weight easy classification examples.
        - Focus the classifier head on harder authentic vs tampered cases.

    Notes:
        The class weights computed above are passed as alpha to preserve the original balancing strategy.
    """
    def __init__(self, alpha=None, gamma=2.0):
        """
        Purpose:
            Store focal-loss hyperparameters for later use.

        Inputs:
            alpha (torch.Tensor | None): Optional class weights for the cross-entropy term.
            gamma (float): Focusing parameter that down-weights easy examples.

        Returns:
            None.

        Notes:
            The implementation is preserved exactly from the source notebook.
        """
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, labels):
        """
        Purpose:
            Compute the focal classification loss for a batch of logits.

        Inputs:
            logits (torch.Tensor): Raw classifier outputs.
            labels (torch.Tensor): Ground-truth image-level labels.

        Returns:
            torch.Tensor: Scalar focal loss.

        Notes:
            Cross-entropy is reweighted by confidence so hard samples contribute more strongly.
        """
        ce = F.cross_entropy(logits, labels, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()

def dice_loss(pred, target, eps=1e-7):
    """
    Purpose:
        Compute a soft Dice loss for segmentation logits.

    Inputs:
        pred (torch.Tensor): Raw segmentation logits.
        target (torch.Tensor): Ground-truth binary masks.
        eps (float): Small constant for numerical stability.

    Returns:
        torch.Tensor: Scalar Dice loss.

    Notes:
        Dice loss emphasizes overlap quality, which is important for localizing manipulated regions.
    """
    prob = torch.sigmoid(pred)
    inter = (prob * target).sum(dim=(1,2,3))
    union = prob.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2.0 * inter + eps) / (union + eps)
    return 1 - dice.mean()

def dice_coef(pred, target, eps=1e-7):
    """
    Purpose:
        Report the Dice coefficient for thresholded segmentation predictions.

    Inputs:
        pred (torch.Tensor): Raw segmentation logits.
        target (torch.Tensor): Ground-truth binary masks.
        eps (float): Small constant for numerical stability.

    Returns:
        float: Mean Dice score across the batch.

    Notes:
        Thresholding at 0.5 converts probability maps into binary tampering masks for reporting.
    """
    prob = torch.sigmoid(pred)
    pred_bin = (prob > 0.5).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3))
    dice = (2.0 * inter + eps) / (union + eps)
    return dice.mean().item()

criterion_cls = FocalLoss(alpha=class_weights)
bce_loss      = nn.BCEWithLogitsLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

ALPHA = 1.5
BETA  = 1.0


### 8.8 Training History and Metric Helpers

Initializes the training history dictionary and defines IoU and F1 scoring functions
used for validation reporting (these do not affect training decisions).

In [ ]:
# ================== 7) Training history and metrics ==================
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": [],
    "val_dice": [],
    "val_iou": [],
    "val_f1": [],
}
best_epoch = 0

def iou_coef(pred, target, eps=1e-7):
    """
    Purpose:
        Compute Intersection over Union for thresholded segmentation masks.

    Inputs:
        pred (torch.Tensor): Raw segmentation logits.
        target (torch.Tensor): Ground-truth binary masks.
        eps (float): Small constant for numerical stability.

    Returns:
        float: Mean IoU across the batch.

    Notes:
        IoU is stricter than Dice and helps quantify how tightly predicted tampered regions match the target mask.
    """
    prob = torch.sigmoid(pred)
    pred_bin = (prob > 0.5).float()
    inter = (pred_bin * target).sum(dim=(1,2,3))
    union = pred_bin.sum(dim=(1,2,3)) + target.sum(dim=(1,2,3)) - inter
    return ((inter + eps) / (union + eps)).mean().item()

def f1_coef(pred, target, eps=1e-7):
    """
    Purpose:
        Compute the F1 score for thresholded segmentation masks.

    Inputs:
        pred (torch.Tensor): Raw segmentation logits.
        target (torch.Tensor): Ground-truth binary masks.
        eps (float): Small constant for numerical stability.

    Returns:
        float: Mean F1 score across the batch.

    Notes:
        F1 summarizes precision and recall for the predicted tampered region.
    """
    prob = torch.sigmoid(pred)
    pred_bin = (prob > 0.5).float()
    tp = (pred_bin * target).sum(dim=(1,2,3))
    fp = (pred_bin * (1.0 - target)).sum(dim=(1,2,3))
    fn = ((1.0 - pred_bin) * target).sum(dim=(1,2,3))
    precision = (tp + eps) / (tp + fp + eps)
    recall = (tp + eps) / (tp + fn + eps)
    return (2.0 * precision * recall / (precision + recall + eps)).mean().item()


### 8.9 Training and Evaluation Routines

`train_one_epoch` trains the model for one pass over the training split using the combined
classification and segmentation loss.

`evaluate` computes loss, accuracy, Dice, IoU, and F1 on a given dataloader.

In [ ]:
def train_one_epoch(epoch):
    """
    Purpose:
        Train the effective submission model for one epoch.

    Inputs:
        epoch (int): Current epoch number for logging.

    Returns:
        tuple: Mean training loss and image-level training accuracy.

    Notes:
        The shared model learns image-level tamper detection and pixel-level localization jointly.
    """
    model.train()
    running_loss = 0.0
    correct = 0

    for images, masks, labels in train_loader:
        images = images.to(device)
        masks  = masks.to(device)
        labels = labels.to(device)

        # Reset gradients before processing the next batch.
        optimizer.zero_grad()

        # Forward pass: compute image-level logits and segmentation logits for the current batch.
        cls_logits, seg_logits = model(images)

        # Classification loss supervises authentic vs tampered prediction.
        loss_cls = criterion_cls(cls_logits, labels)
        # Segmentation loss combines BCE and Dice so the model learns both pixel accuracy and region overlap.
        loss_seg = 0.5 * bce_loss(seg_logits, masks) + 0.5 * dice_loss(seg_logits, masks)
        # Combine both objectives using the preserved assignment weights.
        loss = ALPHA * loss_cls + BETA * loss_seg

        # Backpropagation: compute gradients through the shared backbone and both output heads.
        loss.backward()
        # Clip gradients to stabilize training without changing the architecture or objective.
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        # Optimizer step: update model parameters.
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(cls_logits, dim=1)
        correct += (preds == labels).sum().item()

    # Advance the learning-rate schedule once per epoch.
    scheduler.step()

    epoch_loss = running_loss / len(train_dataset)
    epoch_acc = correct / len(train_dataset)
    print(f"Epoch {epoch:02d} | Train Loss: {epoch_loss:.4f} | Train Acc: {epoch_acc:.4f}")
    return epoch_loss, epoch_acc

def evaluate(epoch, loader, name="Val", return_details=False):
    """
    Purpose:
        Evaluate the trained model on validation or test data.

    Inputs:
        epoch (int): Epoch identifier used for logging.
        loader (DataLoader): Validation or test dataloader.
        name (str): Name of the evaluated split.
        return_details (bool): Whether to return a dictionary of metrics instead of only accuracy.

    Returns:
        dict | float: Detailed metrics when requested, otherwise image-level accuracy.

    Notes:
        Dice, IoU, and F1 quantify segmentation quality, while accuracy summarizes image-level tamper detection.
    """
    model.eval()
    running_loss = 0.0
    correct = 0
    dice_sum = 0.0
    iou_sum = 0.0
    f1_sum = 0.0
    num_batches = 0

    with torch.no_grad():
        for images, masks, labels in loader:
            images = images.to(device)
            masks  = masks.to(device)
            labels = labels.to(device)

            # Forward pass: reuse the model in evaluation mode without gradient tracking.
            cls_logits, seg_logits = model(images)

            # Compute the same multi-task losses used during training for consistent reporting.
            loss_cls = criterion_cls(cls_logits, labels)
            loss_seg = 0.5 * bce_loss(seg_logits, masks) + 0.5 * dice_loss(seg_logits, masks)
            loss = ALPHA * loss_cls + BETA * loss_seg

            running_loss += loss.item() * images.size(0)
            preds = torch.argmax(cls_logits, dim=1)
            correct += (preds == labels).sum().item()

            # Convert segmentation logits into thresholded masks for localization metrics.
            dice_sum += dice_coef(seg_logits, masks)
            iou_sum += iou_coef(seg_logits, masks)
            f1_sum += f1_coef(seg_logits, masks)
            num_batches += 1

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)
    epoch_dice = dice_sum / max(num_batches, 1)
    epoch_iou = iou_sum / max(num_batches, 1)
    epoch_f1 = f1_sum / max(num_batches, 1)

    print(
        f"Epoch {epoch:02d} | {name} Loss: {epoch_loss:.4f} | {name} Acc: {epoch_acc:.4f} | "
        f"{name} Dice: {epoch_dice:.4f} | {name} IoU: {epoch_iou:.4f} | {name} F1: {epoch_f1:.4f}"
    )

    metrics = {
        "loss": epoch_loss,
        "acc": epoch_acc,
        "dice": epoch_dice,
        "iou": epoch_iou,
        "f1": epoch_f1,
    }
    return metrics if return_details else epoch_acc


### 8.10 Training Loop Execution

Runs the effective submission training loop for 50 epochs.
Checkpoints the model whenever validation accuracy improves.
Logs metrics to W&B when experiment tracking is active.

In [ ]:
# ================== 8) Run the effective training loop ==================
NUM_EPOCHS = 50
best_val_acc = 0.0
best_model_path = "/kaggle/working/best_model.pth"

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc = train_one_epoch(epoch)
    val_metrics = evaluate(epoch, val_loader, name="Val", return_details=True)
    val_acc = val_metrics["acc"]

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_metrics["loss"])
    history["val_acc"].append(val_metrics["acc"])
    history["val_dice"].append(val_metrics["dice"])
    history["val_iou"].append(val_metrics["iou"])
    history["val_f1"].append(val_metrics["f1"])

    if WANDB_ACTIVE:
        wandb.log({
            "epoch": epoch,
            "train/loss": train_loss,
            "train/accuracy": train_acc,
            "val/loss": val_metrics["loss"],
            "val/accuracy": val_metrics["acc"],
            "val/dice": val_metrics["dice"],
            "val/iou": val_metrics["iou"],
            "val/f1": val_metrics["f1"],
        })

    if val_acc > best_val_acc:
        # Save the checkpoint with the best image-level validation accuracy, matching the original selection rule.
        best_val_acc = val_acc
        best_epoch = epoch
        torch.save(model.state_dict(), best_model_path)
        print(f"==> Saved new best model (Val Acc: {best_val_acc:.4f})")

    torch.cuda.empty_cache()

print("Training finished. Best Val Acc:", best_val_acc)


### 8.11 Final Test Evaluation

Loads the best checkpoint and evaluates it on the held-out test split.
Stores the results in `TRAINING_HISTORY` and `FINAL_TEST_METRICS` for downstream reporting.

In [ ]:
# ================== 9) Evaluate the best checkpoint on the test split ==================
model.load_state_dict(torch.load(best_model_path, map_location=device))
test_metrics = evaluate(0, test_loader, name="Test", return_details=True)
TRAINING_HISTORY = history
FINAL_TEST_METRICS = test_metrics
print("Final Test Acc:", test_metrics["acc"])
print(
    f"Final Test Dice: {test_metrics['dice']:.4f} | "
    f"IoU: {test_metrics['iou']:.4f} | F1: {test_metrics['f1']:.4f}"
)

if WANDB_ACTIVE:
    wandb.summary.update({
        "best_val_acc": best_val_acc,
        "best_epoch": best_epoch,
        "test/accuracy": test_metrics["acc"],
        "test/dice": test_metrics["dice"],
        "test/iou": test_metrics["iou"],
        "test/f1": test_metrics["f1"],
    })

### 8.1 Training Curves and Learning Behavior

After the effective submission run finishes, the next cell summarizes optimization trends and validation quality through curves.
These plots complement the final metrics by showing how loss, image-level accuracy, and segmentation quality evolve over time.

**Assignment alignment:** This subsection provides supporting evidence that the preserved training setup converges for the assignment task.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

history_df = pd.DataFrame(TRAINING_HISTORY)
display(history_df.tail())

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot training vs validation loss to show overall optimization behavior.
axes[0].plot(history_df.index + 1, history_df["train_loss"], label="Train Loss")
axes[0].plot(history_df.index + 1, history_df["val_loss"], label="Val Loss")
axes[0].set_title("Loss Curves")
axes[0].set_xlabel("Epoch")
axes[0].legend()

# Plot localization metrics to summarize mask-quality trends during validation.
axes[1].plot(history_df.index + 1, history_df["val_dice"], label="Dice")
axes[1].plot(history_df.index + 1, history_df["val_iou"], label="IoU")
axes[1].plot(history_df.index + 1, history_df["val_f1"], label="F1")
axes[1].set_title("Validation Segmentation Metrics")
axes[1].set_xlabel("Epoch")
axes[1].legend()

# Plot image-level accuracy to show tamper-detection performance across epochs.
axes[2].plot(history_df.index + 1, history_df["train_acc"], label="Train Acc")
axes[2].plot(history_df.index + 1, history_df["val_acc"], label="Val Acc")
axes[2].set_title("Image-Level Detection Accuracy")
axes[2].set_xlabel("Epoch")
axes[2].legend()

plt.tight_layout()
plt.show()

if WANDB_ACTIVE:
    wandb.log({"training_curves": wandb.Image(fig)})

## 9. Evaluation Methodology

This notebook evaluates two related tasks:

- **Image-level tamper detection:** based on the classifier head output and reported through accuracy.
- **Pixel-level localization:** based on the segmentation output thresholded at `0.5`.

The reporting metrics are:

- **Dice score:** overlap-focused segmentation quality measure.
- **IoU:** intersection over union between predicted and target masks.
- **F1 score:** harmonic mean of precision and recall on the thresholded segmentation mask.

The added IoU and F1 are reporting metrics only. They do not change optimizer behavior, loss design, or checkpoint selection.

**Assignment alignment:** This section provides the quantitative evidence required to judge both detection quality and localization quality.

In [ ]:
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()


In [ ]:
def denormalize(img_tensor):
    """
    Purpose:
        Convert a normalized image tensor back to displayable RGB space.

    Inputs:
        img_tensor (torch.Tensor): Normalized image tensor in CHW format.

    Returns:
        torch.Tensor: Image tensor scaled back to the [0, 1] range for visualization.

    Notes:
        This helper reverses the ImageNet normalization used during preprocessing.
    """
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img = img_tensor.cpu() * std + mean
    img = torch.clamp(img, 0, 1)
    return img

## 10. Visualization of Predictions

The following cells collect authentic and tampered examples from the test loader and visualize model outputs.
For assignment submission, the notebook makes the qualitative evidence clearer by surfacing:

- original image
- ground-truth mask
- predicted mask
- overlay highlighting predicted manipulated regions

Together, these views show both detection and localization performance.

**Assignment alignment:** This section provides qualitative evidence that the model detects tampering and highlights the manipulated region.

### 10.1 Sample Collection for Qualitative Review

The next helper functions collect balanced authentic and tampered examples from the test split and convert predictions into reviewer-friendly visualizations.
This makes it easier to compare image-level decisions with the corresponding predicted tampering regions.

**Assignment alignment:** This subsection connects the evaluation outputs to qualitative evidence for both detection and localization.

In [ ]:
def collect_samples(loader, num_real=5, num_fake=5):
    """
    Purpose:
        Gather a balanced set of authentic and tampered examples for qualitative visualization.

    Inputs:
        loader (DataLoader): Dataloader that provides images, masks, and image-level labels.
        num_real (int): Number of authentic examples to collect.
        num_fake (int): Number of tampered examples to collect.

    Returns:
        tuple: Two lists containing authentic samples and tampered samples.

    Notes:
        Each sample stores the input image, ground-truth mask, predicted mask probabilities, and image-level labels.
    """
    real_samples = []
    fake_samples = []

    with torch.no_grad():
        for images, masks, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            # Forward pass: compute both image-level and pixel-level predictions for qualitative review.
            cls_logits, seg_logits = model(images)
            seg_probs = torch.sigmoid(seg_logits)
            preds = torch.argmax(cls_logits, dim=1)

            for i in range(images.size(0)):
                sample = {
                    "image": images[i].cpu(),
                    "mask_true": masks[i].cpu(),
                    "mask_pred": seg_probs[i].cpu(),
                    "label": int(labels[i].item()),
                    "pred": int(preds[i].item())
                }

                if sample["label"] == 0 and len(real_samples) < num_real:
                    real_samples.append(sample)

                if sample["label"] == 1 and len(fake_samples) < num_fake:
                    fake_samples.append(sample)

                if len(real_samples) >= num_real and len(fake_samples) >= num_fake:
                    return real_samples, fake_samples

    return real_samples, fake_samples


real_samples, fake_samples = collect_samples(test_loader, 5, 5)

print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_samples_with_masks(samples, title):
    """
    Purpose:
        Visualize predicted tampered regions as red overlays on top of the original image.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the panel.

    Returns:
        None.

    Notes:
        Overlay views help the reader see where the model believes manipulation evidence is concentrated.
    """
    n = len(samples)
    plt.figure(figsize=(4*n, 4))

    for i, s in enumerate(samples):
        img = denormalize(s["image"]).permute(1,2,0).numpy()
        # Threshold the predicted probability map to produce a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Color predicted tampered pixels in red so the localization output is easy to interpret.
        overlay[mask_pred==1] = [1, 0, 0]

        blended = (0.6 * img + 0.4 * overlay)

        plt.subplot(1, n, i+1)
        plt.imshow(blended)
        lbl = "Real" if s["label"]==0 else "Fake"
        pred_lbl = "Real" if s["pred"]==0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

    plt.suptitle(title, fontsize=16)
    plt.tight_layout()
    plt.show()


show_samples_with_masks(real_samples, "5 Real Images with Predicted Manipulation Regions")
show_samples_with_masks(fake_samples, "5 Fake Images with Predicted Manipulation Regions")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display each image together with its predicted binary tampering mask.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample is shown in two rows: the original image first, then the thresholded predicted mask.
    """
    n = len(samples)
    plt.figure(figsize=(6*n, 6))

    for idx, s in enumerate(samples):
        # Convert the normalized tensor back to a displayable RGB image.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to obtain a black-and-white mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Top row: original image with image-level ground truth and prediction labels.
        plt.subplot(2, n, idx+1)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}")
        plt.axis("off")

        # Bottom row: predicted binary mask for tampered-region localization.
        plt.subplot(2, n, n + idx + 1)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Predicted Mask")
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
show_image_and_mask(real_samples, "5 Real Images + Predicted Masks")
show_image_and_mask(fake_samples, "5 Fake Images + Predicted Masks")


In [ ]:
real_samples, fake_samples = collect_samples(test_loader, num_real=10, num_fake=10)
print("Collected Real:", len(real_samples), " Fake:", len(fake_samples))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import math

def show_image_and_mask(samples, title):
    """
    Purpose:
        Display image and predicted-mask pairs in a grid that limits each row to three samples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.

    Returns:
        None.

    Notes:
        Each sample occupies two neighboring columns: one for the image and one for the predicted mask.
    """
    total = len(samples)
    cols = 3
    rows = math.ceil(total / cols)

    plt.figure(figsize=(cols * 6, rows * 4))

    for idx, s in enumerate(samples):
        row = idx // cols
        col = idx % cols

        # Convert the normalized tensor back to RGB for display.
        img = denormalize(s["image"]).permute(1,2,0).numpy()

        # Threshold the predicted probabilities to form a binary tampering mask.
        mask_pred = (s["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        # Compute the first subplot column used by this sample inside the current row.
        base_col = col * 2

        img_pos  = row * cols * 2 + base_col + 1
        mask_pos = img_pos + 1

        # Show the original image and the image-level prediction.
        plt.subplot(rows, cols*2, img_pos)
        plt.imshow(img)
        lbl = "Real" if s["label"] == 0 else "Fake"
        pred_lbl = "Real" if s["pred"] == 0 else "Fake"
        plt.title(f"{lbl} | Pred: {pred_lbl}", fontsize=10)
        plt.axis("off")

        # Show the corresponding thresholded localization mask.
        plt.subplot(rows, cols*2, mask_pos)
        plt.imshow(mask_pred, cmap="gray")
        plt.title("Mask", fontsize=10)
        plt.axis("off")

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()

In [ ]:
show_image_and_mask(real_samples, "10 Real Images (3 per row)")
show_image_and_mask(fake_samples, "10 Fake Images (3 per row)")


### 10.2 Submission-Ready Prediction Panels

The final visualization block packages the qualitative outputs into a compact four-panel format.
Each row aligns the original image, ground-truth mask, predicted mask, and overlay so the assignment reviewer can inspect localization quality quickly.

**Assignment alignment:** This subsection supports the final qualitative presentation requirement of the assignment.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def show_submission_prediction_grid(samples, title, max_items=6):
    """
    Purpose:
        Create a submission-style four-panel view for a small set of qualitative examples.

    Inputs:
        samples (list): Collection of sample dictionaries returned by collect_samples.
        title (str): Figure title shown above the grid.
        max_items (int): Maximum number of examples to visualize.

    Returns:
        matplotlib.figure.Figure | None: The generated figure when samples are available.

    Notes:
        Each row shows the original image, ground-truth mask, predicted mask, and overlay for side-by-side review.
    """
    chosen = samples[:max_items]
    if not chosen:
        print(f"No samples available for: {title}")
        return None

    fig, axes = plt.subplots(len(chosen), 4, figsize=(16, 4 * len(chosen)))
    if len(chosen) == 1:
        axes = np.expand_dims(axes, axis=0)

    column_titles = ["Original Image", "Ground Truth Mask", "Predicted Mask", "Overlay"]

    for row_idx, sample in enumerate(chosen):
        img = denormalize(sample["image"]).permute(1, 2, 0).numpy()
        gt_mask = (sample["mask_true"][0].numpy() > 0.5).astype(np.uint8)
        pred_mask = (sample["mask_pred"][0].numpy() > 0.5).astype(np.uint8)

        overlay = img.copy()
        # Highlight predicted tampered pixels in red to make the localization decision easy to inspect.
        overlay[pred_mask == 1] = [1, 0, 0]
        blended = 0.6 * img + 0.4 * overlay

        panels = [img, gt_mask, pred_mask, blended]
        cmaps = [None, "gray", "gray", None]

        for col_idx, panel in enumerate(panels):
            ax = axes[row_idx, col_idx]
            if cmaps[col_idx] is None:
                ax.imshow(panel)
            else:
                ax.imshow(panel, cmap=cmaps[col_idx])
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(column_titles[col_idx])

        lbl = "Authentic" if sample["label"] == 0 else "Tampered"
        pred_lbl = "Authentic" if sample["pred"] == 0 else "Tampered"
        axes[row_idx, 0].set_ylabel(f"GT: {lbl}\nPred: {pred_lbl}", fontsize=10)

    plt.suptitle(title, fontsize=18)
    plt.tight_layout()
    plt.show()
    return fig

real_fig = show_submission_prediction_grid(real_samples, "Submission Grid: Authentic Image Predictions", max_items=4)
fake_fig = show_submission_prediction_grid(fake_samples, "Submission Grid: Tampered Image Predictions", max_items=4)

if WANDB_ACTIVE:
    if real_fig is not None:
        wandb.log({"submission_grid_authentic": wandb.Image(real_fig)})
    if fake_fig is not None:
        wandb.log({"submission_grid_tampered": wandb.Image(fake_fig)})

## Conclusion

This notebook now reads as a cleaner assignment submission while preserving the original implementation.

**Strengths**
- one Colab notebook covers setup, training, evaluation, and qualitative inspection
- the model performs both image-level detection and pixel-level localization
- experiment tracking is integrated for reproducibility and presentation
- qualitative panels clearly show the relationship between the input image, the ground-truth mask, and the prediction

**Limitations**
- the notebook remains a source-preserving artifact and still contains an earlier duplicated experiment block
- robustness evaluation is limited compared with a full research benchmark
- broader validation and deployment concerns remain outside this assignment scope

**Assignment alignment summary**
- environment and dataset preparation are documented for reproducibility
- the shared model architecture is documented for both detection and localization
- quantitative metrics cover image-level and pixel-level performance
- qualitative figures provide direct evidence of localized tampering predictions

**Submission takeaway**
The notebook fulfills the core assignment objectives: it detects tampered images, localizes manipulated regions, documents the pipeline clearly, and provides both quantitative and qualitative evidence in one Google Colab notebook.